# Gold Match Tickets

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
gold_match_tickets_df = pd.read_csv('../raw/gold_match_tickets.csv', sep=',', on_bad_lines='skip')
gold_match_tickets_df.info()
gold_match_tickets_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   match_id             71 non-null     str  
 1   tickets_sold_b2c     71 non-null     int64
 2   tickets_sold_b2b     71 non-null     int64
 3   tickets_sold_total   71 non-null     int64
 4   seasonpass_holders   71 non-null     int64
 5   tickets_trib1        71 non-null     int64
 6   tickets_trib2_thuis  71 non-null     int64
 7   tickets_trib2_uit    71 non-null     int64
 8   tickets_trib3        71 non-null     int64
 9   tickets_trib4        71 non-null     int64
dtypes: int64(9), str(1)
memory usage: 5.7 KB


,match_id,tickets_sold_b2c,tickets_sold_b2b,tickets_sold_total,seasonpass_holders,tickets_trib1,tickets_trib2_thuis,tickets_trib2_uit,tickets_trib3,tickets_trib4
0,3kewmr690bhoid47mku9wfgno,1095,882,5988,4405,2088,427,1,1903,1569
1,3nrkjdhpc5zse1sdzt1onor2s,1807,953,6652,4405,2256,517,1,2307,1571
2,3rwouz4640n6ngfbwrwsqkp3o,2793,880,7385,4405,2405,572,17,2821,1570
3,3vk3btgbt7w7qy9hjp3f2934k,1981,1003,6929,4405,2441,584,0,2334,1570
4,3yp7glzujzj8nc07qp0sd0kr8,1936,925,6901,4405,2329,566,0,2437,1569


## Dealing with NULL and duplicate values

In [3]:
print("Duplicate rows:", gold_match_tickets_df.duplicated().sum())
print("Duplicate match_ids:", gold_match_tickets_df['match_id'].duplicated().sum())

Duplicate rows: 0
Duplicate match_ids: 0


## Handling Missing Values

In [4]:
# All other columns should be integers – confirm and convert if needed
numeric_cols = gold_match_tickets_df.columns.drop('match_id')
gold_match_tickets_df[numeric_cols] = gold_match_tickets_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

```pd.to_numeric``` with errors='coerce' will turn any non-numeric values into NaN, which you'll then need to handle in the next step.

In [5]:
gold_match_tickets_df.fillna(0, inplace=True)

,match_id,tickets_sold_b2c,tickets_sold_b2b,tickets_sold_total,seasonpass_holders,tickets_trib1,tickets_trib2_thuis,tickets_trib2_uit,tickets_trib3,tickets_trib4
0,3kewmr690bhoid47mku9wfgno,1095,882,5988,4405,2088,427,1,1903,1569
1,3nrkjdhpc5zse1sdzt1onor2s,1807,953,6652,4405,2256,517,1,2307,1571
2,3rwouz4640n6ngfbwrwsqkp3o,2793,880,7385,4405,2405,572,17,2821,1570
3,3vk3btgbt7w7qy9hjp3f2934k,1981,1003,6929,4405,2441,584,0,2334,1570
4,3yp7glzujzj8nc07qp0sd0kr8,1936,925,6901,4405,2329,566,0,2437,1569
...,...,...,...,...,...,...,...,...,...,...
66,ecz9iyxyc5ira9n59m8cp3bis,2967,1079,7473,4235,2485,795,0,2625,1568
67,enw1n6mhkzvb6uptwhzs4gowk,2019,1601,6720,4235,2124,795,0,2225,1576
68,eqncupz92qy89z9bxks1e0t90,2547,2823,7355,4235,2095,776,0,2916,1568
69,ewgb5fczpfa71pfzihlip6yok,2598,2213,7118,4235,2225,795,0,2521,1577


## Validate Data Integrity

In [6]:
# Calculate tribune_sum to validate against tickets_sold_total
gold_match_tickets_df['tribune_sum'] = (
    gold_match_tickets_df['tickets_trib1'] +
    gold_match_tickets_df['tickets_trib2_thuis'] +
    gold_match_tickets_df['tickets_trib2_uit'] +
    gold_match_tickets_df['tickets_trib3'] +
    gold_match_tickets_df['tickets_trib4']
)

In [7]:
# Create a DataFrame with both columns for easy comparison
comparison = gold_match_tickets_df[['match_id', 'tickets_sold_total', 'tribune_sum']].copy()
comparison['difference'] = comparison['tickets_sold_total'] - comparison['tribune_sum']
print(comparison.head(10))   # or use .describe() to see summary statistics

                    match_id  tickets_sold_total  tribune_sum  difference
0  3kewmr690bhoid47mku9wfgno                5988         5988           0
1  3nrkjdhpc5zse1sdzt1onor2s                6652         6652           0
2  3rwouz4640n6ngfbwrwsqkp3o                7385         7385           0
3  3vk3btgbt7w7qy9hjp3f2934k                6929         6929           0
4  3yp7glzujzj8nc07qp0sd0kr8                6901         6901           0
5  42wgdmxzlcn9w08umkea3hpg4                7617         7617           0
6  4729fykixfyfwxlbno9zilq8k                7955         7955           0
7  4bjgrdur7wdidfbs58xmjynf8                8132         8132           0
8  4i4bz7rnjs44z8bt8bnftavbo                7181         7181           0
9  4mbxzkhlc9edsor1s0fjsxses                7217         7217           0


In [8]:
# Check for negative values
(gold_match_tickets_df[numeric_cols] < 0).sum()

tickets_sold_b2c       0
tickets_sold_b2b       0
tickets_sold_total     0
seasonpass_holders     0
tickets_trib1          0
tickets_trib2_thuis    0
tickets_trib2_uit      0
tickets_trib3          0
tickets_trib4          0
dtype: int64

In [9]:
# Drop tribune and channel columns — keep only match_id, tickets_sold_total, seasonpass_holders
cols_to_drop = [
    'tickets_sold_b2c',
    'tickets_sold_b2b',
    'tickets_trib1',
    'tickets_trib2_thuis',
    'tickets_trib2_uit',
    'tickets_trib3',
    'tickets_trib4',
    'tribune_sum',          # validation helper, no longer needed
]
gold_match_tickets_df = gold_match_tickets_df.drop(columns=cols_to_drop)

print(gold_match_tickets_df.shape)
gold_match_tickets_df.head()

(71, 3)


,match_id,tickets_sold_total,seasonpass_holders
0,3kewmr690bhoid47mku9wfgno,5988,4405
1,3nrkjdhpc5zse1sdzt1onor2s,6652,4405
2,3rwouz4640n6ngfbwrwsqkp3o,7385,4405
3,3vk3btgbt7w7qy9hjp3f2934k,6929,4405
4,3yp7glzujzj8nc07qp0sd0kr8,6901,4405


In [10]:
# gold_match_tickets_df.to_csv('../cleaned/gold_match_tickets_cleaned.csv', index=False)
# print('Saved.')